In [35]:
import os
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm import tqdm

In [36]:
# config
TARGET_SR = 10000 # performed better than 4k or 22k
CHUNK_DURATION = 4 # original was 6
all_results = []

# check
audio_folder = "../raw_data/audio_and_txt_files/" 
if os.path.exists(audio_folder):
    files = [f for f in os.listdir(audio_folder) if f.endswith('.wav')]
    print(f"{len(files)} files found.")
else:
    print("Folder not found.")
    files = []

920 files found.


In [37]:
# load patient diagnosis mapping
diagnosis_path = "../raw_data/patient_diagnosis.csv" 

if os.path.exists(diagnosis_path):
    # adjust cols names if different
    df_diag = pd.read_csv(diagnosis_path, names=['patient_id', 'diagnosis'])
    
    # create the mapping: { '101': 'COPD', ... }
    
    diag_map = dict(zip(df_diag['patient_id'].astype(str), df_diag['diagnosis'])) # str for safety as p_id from filename is a string
    print(f"mapping created for {len(diag_map)} patients.")
else:
    print(f"diagnosis file not found at {diagnosis_path}. creating empty map.")
    diag_map = {}

# sanity check
print(list(diag_map.items())[:5])

mapping created for 126 patients.
[('101', 'URTI'), ('102', 'Healthy'), ('103', 'Asthma'), ('104', 'COPD'), ('105', 'URTI')]


In [38]:
def extract_features_raw(y, sr):
    feat = {}
    
    # TEMPORAL
    feat['rms_mean'] = float(np.mean(librosa.feature.rms(y=y)))
    feat['zcr_mean'] = float(np.mean(librosa.feature.zero_crossing_rate(y=y)))
    
    # SPECTRAL
    feat['centroid_mean'] = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    feat['rolloff_mean'] = float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)))
    feat['flatness_mean'] = float(np.mean(librosa.feature.spectral_flatness(y=y)))
    feat['flux_mean'] = float(np.mean(librosa.onset.onset_strength(y=y, sr=sr)))
    
    # MFCC (12, ignoring index 0)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13) 
    for i in range(1, 13):
        feat[f'mfcc_{i}'] = float(np.mean(mfccs[i]))
        
    return feat

In [39]:
for f in tqdm(files):
    path = os.path.join(audio_folder, f)
    
    try:
        # load raw audio (no auto-normalization)
        y, sr = sf.read(path, dtype="float32")
        
        # resample
        if sr != TARGET_SR:
            y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
            sr = TARGET_SR

        # apply Mu-Law compression BEFORE peak normalization
        mu = 255
        # make sure y is non-zero to avoid log issues, though log1p handles 0
        y_compressed = np.sign(y) * np.log1p(mu * np.abs(y)) / np.log1p(mu)
        
        # normalization (scaling to [-1, 1] for Librosa consistency)
        # this ensures that even after compression, the signal is in a standard range
        if np.max(np.abs(y_compressed)) > 0:
            y_final = librosa.util.normalize(y_compressed)
        else:
            y_final = y_compressed

        # cleaning - trim silence
        y_final, _ = librosa.effects.trim(y_final)


## SLICING WO OVERLAPPING (WITH OR WITHOUT - CHOOSE)
        # slicing (6s windows)
        #samples_per_chunk = CHUNK_DURATION * TARGET_SR
        #chunks = [y_final[i : i + samples_per_chunk] for i in range(0, len(y_final), samples_per_chunk) 
                  #if len(y_final[i : i + samples_per_chunk]) == samples_per_chunk]

## SLICE W OVERLAPPING (PICK YOUR POISON)
        duration_seconds = CHUNK_DURATION # pre defined
        step_seconds = 2                  # 3 or 2 seconds, depending on the definition before (overlap of 50%). 4 seconds and have less overlap?
        
        samples_per_chunk = duration_seconds * TARGET_SR
        samples_per_step = step_seconds * TARGET_SR
        
        chunks = []
        # loop would be for every 2 or 3 seconds - based on what was defined before
        for start_idx in range(0, len(y_final) - samples_per_chunk + 1, samples_per_step):
            end_idx = start_idx + samples_per_chunk
            chunk = y_final[start_idx : end_idx]
            chunks.append(chunk)
#END OF SLICING
        
        # metadata extraction
        p_id = f.split('_')[0]
        
        for i, chunk in enumerate(chunks):
            features = extract_features_raw(chunk, TARGET_SR)
            features['patient_id'] = p_id
            features['chunk_id'] = i
            features['original_file'] = f
            
            # map diagnosis - if available
            features['diagnosis'] = diag_map.get(p_id, "Unknown")
            
            all_results.append(features)

    except Exception as e:
        print(f"error processing {f}: {e}")

# convert to df
df_final = pd.DataFrame(all_results)
print(f"shape: {df_final.shape}")

100%|█████████████████████████████████████████████████████████████████████████████████| 920/920 [02:30<00:00,  6.10it/s]


shape: (8916, 22)


In [40]:
print(df_final.columns.tolist())

['rms_mean', 'zcr_mean', 'centroid_mean', 'rolloff_mean', 'flatness_mean', 'flux_mean', 'mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 'mfcc_11', 'mfcc_12', 'patient_id', 'chunk_id', 'original_file', 'diagnosis']


In [41]:
print(df_final['diagnosis'].value_counts().head())

diagnosis
COPD              7780
Pneumonia          333
Healthy            310
URTI               205
Bronchiectasis     144
Name: count, dtype: int64


In [42]:
display(df_final.head(15))

,rms_mean,zcr_mean,centroid_mean,rolloff_mean,flatness_mean,flux_mean,mfcc_1,mfcc_2,mfcc_3,mfcc_4,...,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,patient_id,chunk_id,original_file,diagnosis
0,0.734586,0.001471,103.380980,130.476167,0.000068,1.939369,133.555325,45.050655,31.055117,28.510685,...,15.999349,13.810915,9.979599,8.018103,7.275218,8.493723,223,0,223_1b1_Pr_sc_Meditron.wav,COPD
1,0.735493,0.001681,119.941923,164.594047,0.000097,1.991206,147.711498,43.637800,32.914647,28.863335,...,14.296293,12.104002,8.257276,7.270130,7.243630,8.184478,223,1,223_1b1_Pr_sc_Meditron.wav,COPD
2,0.686210,0.001935,107.829130,130.414359,0.000067,1.580583,151.269196,42.120882,34.016339,31.656191,...,14.792543,14.755346,8.228468,6.491997,5.711367,9.255064,223,2,223_1b1_Pr_sc_Meditron.wav,COPD
3,0.664333,0.002114,107.408422,132.639438,0.000097,1.490879,139.863474,40.680070,33.711874,33.029794,...,15.592032,16.822254,9.888798,6.253796,5.791679,9.209698,223,3,223_1b1_Pr_sc_Meditron.wav,COPD
4,0.724992,0.001681,105.570926,128.930973,0.000047,1.612109,144.691474,35.527395,32.037788,30.692539,...,15.319612,14.514699,10.525392,8.516052,7.521252,7.864186,223,4,223_1b1_Pr_sc_Meditron.wav,COPD
5,0.744723,0.001329,101.826397,128.745550,0.000073,1.513243,146.233232,41.368180,33.837321,30.647703,...,14.771913,14.064785,9.457247,7.655868,7.482498,8.652743,223,5,223_1b1_Pr_sc_Meditron.wav,COPD
6,0.687235,0.001879,105.456686,128.621934,0.000059,1.647425,145.084673,45.211388,33.253622,31.738627,...,14.503697,14.369951,9.972780,6.574307,6.335377,9.505226,223,6,223_1b1_Pr_sc_Meditron.wav,COPD
7,0.633077,0.002194,100.806517,113.602650,0.000053,1.577210,141.827516,47.881412,32.418759,34.262541,...,15.160930,14.062956,10.917266,6.317072,4.567845,8.936956,223,7,223_1b1_Pr_sc_Meditron.wav,COPD
8,0.636758,0.001564,92.243967,108.039953,0.000072,1.598196,138.205813,51.669374,33.264860,33.928936,...,15.307188,14.755253,11.197473,6.037932,3.779242,8.908488,223,8,223_1b1_Pr_sc_Meditron.wav,COPD
9,0.642405,0.001403,85.244015,92.031744,0.000030,1.537083,137.003989,51.773277,33.021772,32.476266,...,16.044378,15.774482,10.548290,6.138236,4.813216,8.882441,223,9,223_1b1_Pr_sc_Meditron.wav,COPD


In [43]:
# save CSV to raw_data
df_final.to_csv("../raw_data/xgboost_features_4s_10kHz_compression_before_normalization_w_overlapping.csv", index=False)
print("done")

done
